In [1]:
import json
import os
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

CHUNKS_FILE = "chunks.json"
CHROMA_DB_DIR = "./chroma_db"
COLLECTION_NAME = "academic_papers"

/home/wiz/Drive/PESU/LLMA/proj/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Text Chunks


In [2]:
if not os.path.exists(CHUNKS_FILE):
    print(f"Error: {CHUNKS_FILE} not found. Please run injestion.ipynb first!")
else:
    with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    if not chunks:
        print("No chunks found to index.")
    else:
        print(f"Loaded {len(chunks)} chunks from {CHUNKS_FILE}")

Loaded 526 chunks from chunks.json


# Initialize Embeddings Model


In [3]:
print("Loading HuggingFace Embeddings model...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading HuggingFace Embeddings model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18127.45it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Prepare Document Objects


In [4]:
documents = []
if 'chunks' in locals() and chunks:
    for chunk in chunks:
        # Safely extract the text
        content = chunk.get("text")
        
        # 1. Skip if the content is None or completely empty
        if not content or not str(content).strip():
            continue
            
        # 2. Force it to be a string just to be absolutely safe
        doc = Document(
            page_content=str(content),
            metadata=chunk.get("metadata", {})
        )
        documents.append(doc)
        
    print(f" Prepared {len(documents)} clean documents for embeddings.")

 Prepared 479 clean documents for embeddings.


# Chroma Vector Store


In [5]:
if documents:
    print(f"Initializing Chroma DB at {CHROMA_DB_DIR} and embedding all chunks...")
    
    # Store in chroma
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=CHROMA_DB_DIR
    )
    
    print("\nDoen All chunks embedded and saved to ChromaDB.")
else:
    print("No documents to store.")

Initializing Chroma DB at ./chroma_db and embedding all chunks...

Doen All chunks embedded and saved to ChromaDB.


# Sanity Check Query


In [6]:
if 'vectorstore' in locals():
    print("-" * 50)
    print("Running a quick sanity check query with metadata filtering...")
    
    test_query = "What is the focus of this paper?"
    test_filter = {"year": {"$eq": 2024}} # Testing our mapped metadata
    
    try:
        results = vectorstore.similarity_search(test_query, k=1, filter=test_filter)
        if results:
            print("\nFound result successfully matching year=2024")
            print(f"Source: {results[0].metadata.get('source')}")
            print(f"Content preview: {results[0].page_content[:150]}...")
        else:
            print("No results found for year=2024. Is the p2.pdf data parsed correctly?")
    except Exception as e:
        print(f"Query check failed: {e}")

--------------------------------------------------
Running a quick sanity check query with metadata filtering...

Found result successfully matching year=2024
Source: p2.pdf
Content preview: [40] S. Min, M. Lewis, L. Zettlemoyer, and H. Hajishirzi. Metaicl: Learning to learn in context. arXiv preprint arXiv:2110.15943, 2021....
